
# Banking Stock Market Data Validation

## Project
Financial Market Intelligence & Currency Risk Analytics

## Phase 4: Data Quality Validation

### Business Objective
Validate historical stock-market data for three Indian banking stocks:

- HDFCBANK.BSE — HDFC Bank
- ICICIBANK.BSE — ICICI Bank
- SBIN.BSE — State Bank of India

The validated data will support banking stock performance analysis, price volatility monitoring, and comparative market intelligence.

### Data Source
Alpha Vantage API

### Expected Dataset
- Total records: 300
- Banking stocks: 3
- Expected observations per stock: 100

### Dataset Columns
1. observation_date
2. stock_symbol
3. open_price
4. high_price
5. low_price
6. close_price
7. volume
8. source

## Validation Plan

1. Load stock data and ingestion metadata.
2. Validate schema and data types.
3. Check missing values.
4. Identify duplicate records.
5. Validate OHLC prices and trading volume.
6. Validate stock symbols and source.
7. Check date coverage and stock-date alignment.
8. Screen for statistical outliers.
9. Generate validation reports and processed data.

## Business Importance

Reliable stock-market data enables accurate price-return calculations, volatility analysis, and comparative banking-sector performance reporting.

Banking stock intelligence is a secondary module of this project. It does not directly calculate the INR cost of import payments.

## Data Integrity Principles

- Preserve the original API extract.
- Do not automatically remove unusual market observations.
- Document every validation finding.
- Save processed datasets separately from raw data.
- Maintain reproducibility and auditability.

In [1]:

from pathlib import Path
import sys

# Locate the project root.
project_root = Path.cwd().resolve()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parents[1]
elif project_root.name.lower() == "python":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Python.Scripts.pipeline_logger import get_logger

logger = get_logger("Banking_Stocks_Data_Validation")

logger.info("Banking stocks data validation notebook initialized")


2026-11-02 16:06:57,227 | INFO | Banking_Stocks_Data_Validation | Banking stocks data validation notebook initialized



## 1. Load Banking Stock Data and Metadata

### Business Objective
Load the original Alpha Vantage banking stock dataset and its ingestion metadata for data-quality validation.

### Source Files
- Data/raw/banking_stocks_raw.csv
- Data/raw/banking_stocks_ingestion_metadata.json

### Validation Method
1. Locate the project root directory.
2. Confirm both source files exist.
3. Load stock records into a pandas DataFrame.
4. Convert observation_date to datetime.
5. Load ingestion metadata.
6. Verify the expected schema and record counts.
7. Review stock symbols and data types.

### Data Integrity
The original source files must remain unchanged during validation.


In [1]:

from pathlib import Path
import json
import pandas as pd

# Identify project root
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

# Define source file paths
RAW_DATA_DIR = PROJECT_ROOT / "Data" / "raw"

STOCK_CSV_PATH = (
    RAW_DATA_DIR / "banking_stocks_raw.csv"
)

STOCK_METADATA_PATH = (
    RAW_DATA_DIR / "banking_stocks_ingestion_metadata.json"
)

# Confirm files exist
assert STOCK_CSV_PATH.exists(), (
    f"Stock CSV not found: {STOCK_CSV_PATH}"
)

assert STOCK_METADATA_PATH.exists(), (
    f"Stock metadata not found: {STOCK_METADATA_PATH}"
)

# Load stock dataset
df_stocks_raw = pd.read_csv(
    STOCK_CSV_PATH,
    parse_dates=["observation_date"]
)

# Load ingestion metadata
with open(
    STOCK_METADATA_PATH,
    "r",
    encoding="utf-8"
) as file:
    stock_metadata = json.load(file)

# Define expected schema
REQUIRED_STOCK_COLUMNS = [
    "observation_date",
    "stock_symbol",
    "open_price",
    "high_price",
    "low_price",
    "close_price",
    "volume",
    "source"
]

EXPECTED_STOCKS = {
    "HDFCBANK.BSE",
    "ICICIBANK.BSE",
    "SBIN.BSE"
}

# Initial loading checks
assert set(REQUIRED_STOCK_COLUMNS).issubset(
    df_stocks_raw.columns
), "Required stock columns are missing."

assert len(df_stocks_raw) == 300, (
    "Stock record count does not match expected total."
)

assert set(df_stocks_raw["stock_symbol"].unique()) == (
    EXPECTED_STOCKS
), "Unexpected or missing stock symbols."

# Record counts by stock
stock_counts = (
    df_stocks_raw["stock_symbol"]
    .value_counts()
    .sort_index()
)

assert (stock_counts == 100).all(), (
    "Expected 100 records for each banking stock."
)

print("Banking Stock Data Loading: SUCCESS")
print("Metadata Loading: SUCCESS")

print("\nTotal Records:", len(df_stocks_raw))

print("\nRecords by Stock:")
print(stock_counts)

print("\nData Types:")
print(df_stocks_raw.dtypes)

print("\nMetadata Fields:")
print(list(stock_metadata.keys()))

print("\nFirst Five Records:")
display(df_stocks_raw.head())


Banking Stock Data Loading: SUCCESS
Metadata Loading: SUCCESS

Total Records: 300

Records by Stock:
stock_symbol
HDFCBANK.BSE     100
ICICIBANK.BSE    100
SBIN.BSE         100
Name: count, dtype: int64

Data Types:
observation_date    datetime64[us]
stock_symbol                   str
open_price                 float64
high_price                 float64
low_price                  float64
close_price                float64
volume                       int64
source                         str
dtype: object

Metadata Fields:
['dataset', 'source', 'api_function', 'outputsize', 'stock_symbols', 'retrieved_at_utc', 'total_records', 'records_by_stock', 'earliest_observation', 'latest_observation', 'columns']

First Five Records:


,observation_date,stock_symbol,open_price,high_price,low_price,close_price,volume,source
0,2026-05-22,HDFCBANK.BSE,759.00,774.90,759.00,766.40,629420,Alpha Vantage
1,2026-05-25,HDFCBANK.BSE,778.00,788.00,775.25,786.85,964192,Alpha Vantage
2,2026-05-26,HDFCBANK.BSE,785.95,790.90,777.00,779.00,932064,Alpha Vantage
3,2026-05-27,HDFCBANK.BSE,772.10,772.10,756.60,758.50,4893862,Alpha Vantage
4,2026-05-29,HDFCBANK.BSE,749.00,760.65,738.10,744.75,1799471,Alpha Vantage



## 2. Schema and Data Type Validation

### Business Objective
Verify that the banking stock dataset follows the required structure for stock-market intelligence, financial calculations, and PostgreSQL integration.

### Validation Method
1. Confirm exactly eight columns are present.
2. Verify all required columns exist.
3. Identify unexpected columns.
4. Validate observation_date as datetime.
5. Validate stock_symbol as string-compatible.
6. Validate all OHLC price columns as numeric.
7. Validate volume as integer.
8. Validate source as string-compatible.

### Business Rules
- Observation dates must use datetime format.
- Stock identifiers and data sources must contain text values.
- Open, high, low, and close prices must be numeric.
- Trading volume must be integer-based.
- Unexpected or missing columns must be investigated.

### Business Impact
Incorrect data types may cause calculation errors, database-loading failures, or unreliable Power BI reporting.

In [2]:

from pandas.api.types import (
    is_datetime64_any_dtype,
    is_numeric_dtype,
    is_integer_dtype,
    is_string_dtype
)

# Define expected schema
EXPECTED_COLUMNS = [
    "observation_date",
    "stock_symbol",
    "open_price",
    "high_price",
    "low_price",
    "close_price",
    "volume",
    "source"
]

# Identify missing and unexpected columns
missing_columns = set(EXPECTED_COLUMNS) - set(
    df_stocks_raw.columns
)

unexpected_columns = set(df_stocks_raw.columns) - set(
    EXPECTED_COLUMNS
)

# Define schema validation rules
schema_checks = {
    "Exactly 8 columns": len(df_stocks_raw.columns) == 8,

    "All required columns present": len(missing_columns) == 0,

    "No unexpected columns": len(unexpected_columns) == 0,

    "Observation date is datetime": is_datetime64_any_dtype(
        df_stocks_raw["observation_date"]
    ),

    "Stock symbol is string-compatible": is_string_dtype(
        df_stocks_raw["stock_symbol"]
    ),

    "All OHLC prices are numeric": all(
        is_numeric_dtype(df_stocks_raw[column])
        for column in [
            "open_price",
            "high_price",
            "low_price",
            "close_price"
        ]
    ),

    "Trading volume is integer": is_integer_dtype(
        df_stocks_raw["volume"]
    ),

    "Source is string-compatible": is_string_dtype(
        df_stocks_raw["source"]
    )
}

# Create schema validation report
df_stock_schema_validation = pd.DataFrame([
    {
        "validation_rule": rule,
        "status": "PASS" if passed else "FAIL"
    }
    for rule, passed in schema_checks.items()
])

print("Banking Stock Schema Validation Report:")
display(df_stock_schema_validation)

print("\nMissing Columns:", sorted(missing_columns))
print("Unexpected Columns:", sorted(unexpected_columns))

# Overall validation result
passed_checks = sum(schema_checks.values())
total_checks = len(schema_checks)

print(
    f"\nSchema Checks Passed: "
    f"{passed_checks}/{total_checks}"
)

if all(schema_checks.values()):
    print("Banking Stock Schema Validation: PASSED")
else:
    print("Banking Stock Schema Validation: FAILED")
    print("Investigate schema issues before processing.")


Banking Stock Schema Validation Report:


,validation_rule,status
0,Exactly 8 columns,PASS
1,All required columns present,PASS
2,No unexpected columns,PASS
3,Observation date is datetime,PASS
4,Stock symbol is string-compatible,PASS
5,All OHLC prices are numeric,PASS
6,Trading volume is integer,PASS
7,Source is string-compatible,PASS



Missing Columns: []
Unexpected Columns: []

Schema Checks Passed: 8/8
Banking Stock Schema Validation: PASSED



## 3. Missing Value Validation

### Business Objective
Ensure that banking stock observations contain complete information for financial market analysis.

### Validation Method
1. Count missing values in each column.
2. Calculate missing-value percentages.
3. Identify records containing at least one missing value.
4. Calculate overall dataset completeness.
5. Report validation results.

### Validation Rules
The following columns must not contain missing values:
- observation_date
- stock_symbol
- open_price
- high_price
- low_price
- close_price
- volume
- source

### Business Impact
Missing OHLC prices can affect stock-return and volatility calculations.

Missing trading volume can affect liquidity analysis.

Missing dates or stock symbols can cause inaccurate comparisons between banking stocks.

### Data Integrity
Do not automatically fill missing stock prices or trading volumes. Any missing observations must be investigated before processing.


In [3]:

# Calculate missing values for each column
missing_counts = df_stocks_raw.isna().sum()

# Calculate missing percentages
missing_percentages = (
    df_stocks_raw.isna().mean() * 100
)

# Create missing-value validation report
df_stock_missing_validation = pd.DataFrame({
    "column_name": df_stocks_raw.columns,
    "missing_count": missing_counts.values,
    "missing_percentage": missing_percentages.values
})

# Assign validation status
df_stock_missing_validation["status"] = (
    df_stock_missing_validation["missing_count"]
    .apply(lambda count: "PASS" if count == 0 else "FAIL")
)

# Count records containing missing values
rows_with_missing = int(
    df_stocks_raw.isna().any(axis=1).sum()
)

# Calculate overall dataset completeness
total_cells = df_stocks_raw.size
missing_cells = int(
    df_stocks_raw.isna().sum().sum()
)

completeness_percentage = (
    (total_cells - missing_cells) / total_cells * 100
)

# Display results
print("Banking Stock Missing Value Validation Report:")
display(df_stock_missing_validation)

print("\nTotal Records:", len(df_stocks_raw))
print("Records With Missing Values:", rows_with_missing)
print("Total Missing Cells:", missing_cells)
print(
    f"Dataset Completeness: {completeness_percentage:.2f}%"
)

# Overall validation result
if missing_cells == 0:
    print("\nBanking Stock Missing Value Validation: PASSED")
else:
    print("\nBanking Stock Missing Value Validation: FAILED")
    print("Investigate missing values before processing.")


Banking Stock Missing Value Validation Report:


,column_name,missing_count,missing_percentage,status
0,observation_date,0,0.0,PASS
1,stock_symbol,0,0.0,PASS
2,open_price,0,0.0,PASS
3,high_price,0,0.0,PASS
4,low_price,0,0.0,PASS
5,close_price,0,0.0,PASS
6,volume,0,0.0,PASS
7,source,0,0.0,PASS



Total Records: 300
Records With Missing Values: 0
Total Missing Cells: 0
Dataset Completeness: 100.00%

Banking Stock Missing Value Validation: PASSED



## 4. Duplicate Record Validation

### Business Objective
Ensure each banking stock has one unique observation per trading date.

### Validation Method
1. Identify exact duplicate rows across all columns.
2. Identify duplicate records using the business key:
   - observation_date
   - stock_symbol
3. Count duplicate records and affected business keys.
4. Display duplicate observations for investigation.

### Validation Rules
- Exact duplicate records are not permitted.
- Each stock symbol must have only one record per observation date.
- Duplicate business keys must be investigated before database loading.

### Business Impact
Duplicate stock records can inflate trading volumes, distort time-series analysis, and create misleading financial reports.

In [4]:

# Define the stock business key
STOCK_BUSINESS_KEY = [
    "observation_date",
    "stock_symbol"
]

# Check exact duplicate rows
exact_duplicate_count = int(
    df_stocks_raw.duplicated().sum()
)

# Check duplicate stock-date business keys
business_key_duplicate_count = int(
    df_stocks_raw.duplicated(
        subset=STOCK_BUSINESS_KEY
    ).sum()
)

# Identify all records involved in duplicate business keys
duplicate_key_rows = df_stocks_raw[
    df_stocks_raw.duplicated(
        subset=STOCK_BUSINESS_KEY,
        keep=False
    )
].copy()

# Create duplicate validation report
df_stock_duplicate_validation = pd.DataFrame([
    {
        "validation_rule": "Exact duplicate rows",
        "duplicate_count": exact_duplicate_count,
        "status": "PASS" if exact_duplicate_count == 0 else "FAIL"
    },
    {
        "validation_rule": "Duplicate stock-date business keys",
        "duplicate_count": business_key_duplicate_count,
        "status": "PASS" if business_key_duplicate_count == 0 else "FAIL"
    }
])

print("Banking Stock Duplicate Validation Report:")
display(df_stock_duplicate_validation)

print("\nTotal Records:", len(df_stocks_raw))
print("Exact Duplicate Rows:", exact_duplicate_count)
print(
    "Duplicate Business Keys:",
    business_key_duplicate_count
)
print(
    "Rows Involved in Duplicate Business Keys:",
    len(duplicate_key_rows)
)

# Display problematic records if found
if not duplicate_key_rows.empty:
    print("\nDuplicate Records Requiring Investigation:")
    display(
        duplicate_key_rows.sort_values(STOCK_BUSINESS_KEY)
    )

# Overall validation result
if exact_duplicate_count == 0 and business_key_duplicate_count == 0:
    print("\nBanking Stock Duplicate Validation: PASSED")
else:
    print("\nBanking Stock Duplicate Validation: FAILED")
    print("Investigate duplicate records before processing.")


Banking Stock Duplicate Validation Report:


,validation_rule,duplicate_count,status
0,Exact duplicate rows,0,PASS
1,Duplicate stock-date business keys,0,PASS



Total Records: 300
Exact Duplicate Rows: 0
Duplicate Business Keys: 0
Rows Involved in Duplicate Business Keys: 0

Banking Stock Duplicate Validation: PASSED



## 5. OHLC Price and Trading Volume Validation

### Business Objective
Verify that banking stock prices and trading volumes satisfy fundamental financial-market data rules.

### Validation Method
1. Identify zero or negative OHLC prices.
2. Identify non-finite OHLC prices.
3. Check whether high_price is greater than or equal to open_price, close_price, and low_price.
4. Check whether low_price is less than or equal to open_price, close_price, and high_price.
5. Identify negative trading volumes.
6. Count zero-volume observations separately.
7. Summarize the results by validation rule.

### Business Rules
- All OHLC prices must be positive and finite.
- High price must be the maximum or tied maximum among OHLC prices.
- Low price must be the minimum or tied minimum among OHLC prices.
- Trading volume must not be negative.
- Zero trading volume must be flagged for review, not automatically classified as invalid.

### Business Impact
Invalid prices can distort daily returns, volatility, and stock-performance comparisons.

Incorrect high-low relationships may indicate corrupted market data.

Negative trading volumes are invalid. Zero-volume observations may require further investigation.

In [5]:

import numpy as np

# Define OHLC price columns
PRICE_COLUMNS = [
    "open_price",
    "high_price",
    "low_price",
    "close_price"
]

# Create a validation working copy
df_stock_price_check = df_stocks_raw.copy()

# Validate positive prices
invalid_positive_prices = (
    df_stock_price_check[PRICE_COLUMNS] <= 0
).any(axis=1)

# Validate finite prices
invalid_finite_prices = ~np.isfinite(
    df_stock_price_check[PRICE_COLUMNS]
).all(axis=1)

# Validate high price consistency
invalid_high = (
    df_stock_price_check["high_price"] <
    df_stock_price_check[PRICE_COLUMNS].max(axis=1)
)

# Validate low price consistency
invalid_low = (
    df_stock_price_check["low_price"] >
    df_stock_price_check[PRICE_COLUMNS].min(axis=1)
)

# Validate trading volume
negative_volume = (
    df_stock_price_check["volume"] < 0
)

zero_volume = (
    df_stock_price_check["volume"] == 0
)

# Build validation report
price_checks = {
    "Zero or negative OHLC prices": int(
        invalid_positive_prices.sum()
    ),
    "Non-finite OHLC prices": int(
        invalid_finite_prices.sum()
    ),
    "Invalid high price relationships": int(
        invalid_high.sum()
    ),
    "Invalid low price relationships": int(
        invalid_low.sum()
    ),
    "Negative trading volumes": int(
        negative_volume.sum()
    )
}

df_stock_price_validation = pd.DataFrame([
    {
        "validation_rule": rule,
        "invalid_records": count,
        "status": "PASS" if count == 0 else "FAIL"
    }
    for rule, count in price_checks.items()
])

print("Banking Stock OHLC and Volume Validation Report:")
display(df_stock_price_validation)

print("\nZero-Volume Observations:", int(zero_volume.sum()))

# Identify rows failing any mandatory price or volume rule
invalid_record_mask = (
    invalid_positive_prices |
    invalid_finite_prices |
    invalid_high |
    invalid_low |
    negative_volume
)

df_invalid_stock_records = df_stock_price_check[
    invalid_record_mask
].copy()

print("Records Failing Mandatory Rules:", len(df_invalid_stock_records))

if not df_invalid_stock_records.empty:
    print("\nRecords Requiring Investigation:")
    display(df_invalid_stock_records)

# Overall validation result
if all(count == 0 for count in price_checks.values()):
    print("\nBanking Stock OHLC and Volume Validation: PASSED")
else:
    print("\nBanking Stock OHLC and Volume Validation: FAILED")
    print("Investigate invalid financial values before processing.")


Banking Stock OHLC and Volume Validation Report:


,validation_rule,invalid_records,status
0,Zero or negative OHLC prices,0,PASS
1,Non-finite OHLC prices,0,PASS
2,Invalid high price relationships,0,PASS
3,Invalid low price relationships,0,PASS
4,Negative trading volumes,0,PASS



Zero-Volume Observations: 0
Records Failing Mandatory Rules: 0

Banking Stock OHLC and Volume Validation: PASSED



## 6. Banking Stock Symbol and Data Source Validation

### Business Objective
Ensure that the dataset contains only the approved Indian banking stocks and the expected API data source.

### Approved Stocks
- HDFCBANK.BSE — HDFC Bank
- ICICIBANK.BSE — ICICI Bank
- SBIN.BSE — State Bank of India

### Data Source
Alpha Vantage

### Validation Method
1. Verify that every stock symbol is approved.
2. Verify that all three expected stocks are present.
3. Confirm the data source is Alpha Vantage.
4. Check that each stock has 100 observations.
5. Reconcile stock counts with ingestion metadata.

### Business Impact
Incorrect stock symbols can lead to inaccurate banking-sector comparisons.

Incorrect source information can affect data lineage and auditability.

Record-count mismatches may indicate incomplete ingestion or data loss.


In [6]:

# Define approved stock symbols and source
APPROVED_STOCKS = {
    "HDFCBANK.BSE",
    "ICICIBANK.BSE",
    "SBIN.BSE"
}

APPROVED_STOCK_SOURCE = "Alpha Vantage"

# Calculate actual record counts
actual_stock_counts = (
    df_stocks_raw["stock_symbol"]
    .value_counts()
    .sort_index()
)

# Extract expected counts from ingestion metadata
metadata_stock_counts = stock_metadata["records_by_stock"]

# Define validation checks
stock_source_checks = {
    "Only approved stock symbols": (
        df_stocks_raw["stock_symbol"]
        .isin(APPROVED_STOCKS)
        .all()
    ),

    "All three approved stocks present": (
        set(df_stocks_raw["stock_symbol"].unique())
        == APPROVED_STOCKS
    ),

    "Approved Alpha Vantage source": (
        df_stocks_raw["source"]
        .eq(APPROVED_STOCK_SOURCE)
        .all()
    ),

    "100 observations per stock": (
        actual_stock_counts.eq(100).all()
    ),

    "Metadata total records match": (
        len(df_stocks_raw)
        == int(stock_metadata["total_records"])
    ),

    "Metadata stock counts match": all(
        int(actual_stock_counts.get(symbol, 0))
        == int(metadata_stock_counts.get(symbol, -1))
        for symbol in APPROVED_STOCKS
    )
}

# Build validation report
df_stock_source_validation = pd.DataFrame([
    {
        "validation_rule": rule,
        "status": "PASS" if passed else "FAIL"
    }
    for rule, passed in stock_source_checks.items()
])

print("Banking Stock Symbol and Source Validation Report:")
display(df_stock_source_validation)

print("\nActual Records by Stock:")
display(
    actual_stock_counts.reset_index(
        name="record_count"
    )
)

print("\nMetadata Records by Stock:")
print(metadata_stock_counts)

# Overall validation result
passed_checks = sum(stock_source_checks.values())
total_checks = len(stock_source_checks)

print(f"\nChecks Passed: {passed_checks}/{total_checks}")

if all(stock_source_checks.values()):
    print("Banking Stock Symbol and Source Validation: PASSED")
else:
    print("Banking Stock Symbol and Source Validation: FAILED")
    print("Investigate stock symbols, source, or metadata discrepancies.")


Banking Stock Symbol and Source Validation Report:


,validation_rule,status
0,Only approved stock symbols,PASS
1,All three approved stocks present,PASS
2,Approved Alpha Vantage source,PASS
3,100 observations per stock,PASS
4,Metadata total records match,PASS
5,Metadata stock counts match,PASS



Actual Records by Stock:


,stock_symbol,record_count
0,HDFCBANK.BSE,100
1,ICICIBANK.BSE,100
2,SBIN.BSE,100



Metadata Records by Stock:
{'HDFCBANK.BSE': 100, 'ICICIBANK.BSE': 100, 'SBIN.BSE': 100}

Checks Passed: 6/6
Banking Stock Symbol and Source Validation: PASSED



## 7. Banking Stock Date Range and Trading-Date Alignment Validation

### Business Objective
Verify the historical date coverage and trading-date alignment of HDFCBANK.BSE, ICICIBANK.BSE, and SBIN.BSE.

### Validation Method
1. Identify the earliest and latest observation dates for each stock.
2. Confirm all observations fall within the project analysis period.
3. Count unique trading dates for each stock.
4. Compare trading-date sets across the three stocks.
5. Identify dates missing from individual stocks relative to the others.
6. Identify weekend observations.
7. Compare stock trading dates with the validated FX dataset.

### Validation Rules
- No stock observation may fall outside the project analysis period.
- Each stock must have 100 unique observation dates.
- Weekend observations must be flagged.
- Trading-date differences must be documented.
- FX dates must be available for stock observations used in cross-market analysis.

### Business Impact
Misaligned trading dates can distort comparative stock returns, volatility calculations, and stock-FX correlation analysis.

In [7]:

# Define project analysis period
START_DATE = pd.Timestamp("2026-05-21")
END_DATE = pd.Timestamp("2026-10-13")

# Summarize date coverage by stock
df_stock_date_coverage = (
    df_stocks_raw.groupby("stock_symbol")
    .agg(
        first_date=("observation_date", "min"),
        last_date=("observation_date", "max"),
        observation_count=("observation_date", "count"),
        unique_dates=("observation_date", "nunique")
    )
    .reset_index()
)

print("Banking Stock Date Coverage:")
display(df_stock_date_coverage)

# Check observations outside the project period
outside_range = df_stocks_raw[
    ~df_stocks_raw["observation_date"].between(
        START_DATE, END_DATE
    )
]

# Build date sets for each stock
stock_date_sets = {
    symbol: set(group["observation_date"])
    for symbol, group in df_stocks_raw.groupby("stock_symbol")
}

# Check whether all stock date sets are identical
stock_dates_aligned = (
    len({
        frozenset(dates)
        for dates in stock_date_sets.values()
    }) == 1
)

# Identify missing dates relative to other stocks
all_stock_dates = set().union(*stock_date_sets.values())

missing_relative_dates = {
    symbol: sorted(all_stock_dates - dates)
    for symbol, dates in stock_date_sets.items()
}

# Identify weekend observations
weekend_records = df_stocks_raw[
    df_stocks_raw["observation_date"].dt.dayofweek >= 5
]

# Load validated FX dates for cross-market alignment
FX_PROCESSED_PATH = (
    PROJECT_ROOT / "Data" / "processed"
    / "fx_rates_processed.csv"
)

assert FX_PROCESSED_PATH.exists(), (
    "Validated FX dataset not found."
)

df_fx_processed = pd.read_csv(
    FX_PROCESSED_PATH,
    parse_dates=["observation_date"]
)

fx_dates = set(df_fx_processed["observation_date"])

# Identify stock dates without FX observations
stock_dates_without_fx = sorted(
    all_stock_dates - fx_dates
)

# Build validation report
date_checks = {
    "No dates outside project period": (
        len(outside_range) == 0
    ),
    "100 unique dates per stock": (
        df_stock_date_coverage["unique_dates"].eq(100).all()
    ),
    "No weekend observations": (
        len(weekend_records) == 0
    ),
    "All stock dates covered by FX": (
        len(stock_dates_without_fx) == 0
    )
}

df_stock_date_validation = pd.DataFrame([
    {
        "validation_rule": rule,
        "status": "PASS" if passed else "FAIL"
    }
    for rule, passed in date_checks.items()
])

print("\nBanking Stock Date Validation Report:")
display(df_stock_date_validation)

print("\nStock Trading Dates Fully Aligned:", stock_dates_aligned)

print("\nMissing Dates Relative to Other Stocks:")
for symbol, dates in missing_relative_dates.items():
    print(f"{symbol}: {len(dates)}")
    if dates:
        print([d.strftime("%Y-%m-%d") for d in dates])

print("\nWeekend Records:", len(weekend_records))
print("Dates Outside Project Range:", len(outside_range))
print("Stock Dates Without FX:", len(stock_dates_without_fx))

if stock_dates_without_fx:
    print([
        d.strftime("%Y-%m-%d")
        for d in stock_dates_without_fx
    ])

# Overall validation result
passed_checks = sum(date_checks.values())
total_checks = len(date_checks)

print(f"\nChecks Passed: {passed_checks}/{total_checks}")

if all(date_checks.values()):
    print("Banking Stock Date Coverage Validation: PASSED")
else:
    print("Banking Stock Date Coverage Validation: FAILED")
    print("Investigate date coverage issues before processing.")


Banking Stock Date Coverage:


,stock_symbol,first_date,last_date,observation_count,unique_dates
0,HDFCBANK.BSE,2026-05-22,2026-10-14,100,100
1,ICICIBANK.BSE,2026-05-22,2026-10-14,100,100
2,SBIN.BSE,2026-05-22,2026-10-14,100,100



Banking Stock Date Validation Report:


,validation_rule,status
0,No dates outside project period,FAIL
1,100 unique dates per stock,PASS
2,No weekend observations,PASS
3,All stock dates covered by FX,FAIL



Stock Trading Dates Fully Aligned: True

Missing Dates Relative to Other Stocks:
HDFCBANK.BSE: 0
ICICIBANK.BSE: 0
SBIN.BSE: 0

Weekend Records: 0
Dates Outside Project Range: 3
Stock Dates Without FX: 1
['2026-10-14']

Checks Passed: 2/4
Banking Stock Date Coverage Validation: FAILED
Investigate date coverage issues before processing.



### 7.1 Date Coverage Investigation and Resolution

#### Issue Identified
The Alpha Vantage dataset contains three stock observations dated October 14, 2026, which fall outside the approved project analysis period ending October 13, 2026.

#### Root Cause
The extracted stock dataset extends one trading day beyond the approved analysis period.

#### Resolution
- Preserve all 300 original stock records.
- Exclude observations outside the approved analysis period from the analysis working dataset.
- Retain the excluded records separately for auditability.
- Revalidate the date range and FX alignment.

#### Business Justification
Consistent date boundaries are required for reliable comparisons between banking stock performance and INR exchange-rate movements.

In [8]:

# Investigate observations outside the approved period
df_stock_outside_period = df_stocks_raw[
    ~df_stocks_raw["observation_date"].between(
        START_DATE, END_DATE
    )
].copy()

print("Observations Outside Approved Period:")
display(df_stock_outside_period)

# Create an analysis-period working dataset
df_stocks_analysis = df_stocks_raw[
    df_stocks_raw["observation_date"].between(
        START_DATE, END_DATE
    )
].copy()

df_stocks_analysis = df_stocks_analysis.sort_values(
    ["stock_symbol", "observation_date"]
).reset_index(drop=True)

# Verify scope filtering
assert len(df_stocks_raw) == 300
assert len(df_stock_outside_period) == 3
assert len(df_stocks_analysis) == 297

assert df_stocks_analysis["observation_date"].between(
    START_DATE, END_DATE
).all()

# Verify FX alignment
analysis_stock_dates = set(
    df_stocks_analysis["observation_date"]
)

fx_dates = set(df_fx_processed["observation_date"])

unmatched_dates = sorted(
    analysis_stock_dates - fx_dates
)

assert len(unmatched_dates) == 0

print("\nOriginal Stock Records:", len(df_stocks_raw))
print("Out-of-Period Records:", len(df_stock_outside_period))
print("Analysis Stock Records:", len(df_stocks_analysis))
print("Stock Dates Without FX:", len(unmatched_dates))

print("\nRecords by Stock After Filtering:")
print(df_stocks_analysis["stock_symbol"].value_counts())

print("\nStock Date Range and FX Alignment: PASSED")


Observations Outside Approved Period:


,observation_date,stock_symbol,open_price,high_price,low_price,close_price,volume,source
99,2026-10-14,HDFCBANK.BSE,712.80,718.5,708.00,710.70,1293693,Alpha Vantage
199,2026-10-14,ICICIBANK.BSE,1354.35,1360.8,1339.05,1339.20,534692,Alpha Vantage
299,2026-10-14,SBIN.BSE,956.40,956.7,945.50,954.15,355201,Alpha Vantage



Original Stock Records: 300
Out-of-Period Records: 3
Analysis Stock Records: 297
Stock Dates Without FX: 0

Records by Stock After Filtering:
stock_symbol
HDFCBANK.BSE     99
ICICIBANK.BSE    99
SBIN.BSE         99
Name: count, dtype: int64

Stock Date Range and FX Alignment: PASSED



## 8. Banking Stock Statistical Outlier Screening

### Business Objective
Identify unusual closing-price and trading-volume observations for the three Indian banking stocks.

### Analysis Dataset
Use the 297-record date-aligned working dataset.

### Validation Method
1. Calculate Q1, Q3, and IQR separately for each stock.
2. Apply the 1.5 × IQR rule to closing prices.
3. Apply the same method to trading volumes.
4. Flag unusual observations.
5. Summarize outliers by stock and metric.
6. Preserve all flagged observations for investigation.

### Business Impact
Unusual closing prices may indicate substantial market movements.

Unusual trading volumes may indicate increased market participation, news-driven activity, or other events requiring investigation.

### Important Consideration
The IQR method identifies unusual values relative to the observed distribution. It does not establish that the data is incorrect.

For financial risk analysis, unusual daily percentage returns may be more informative than unusual absolute price levels. Return-based outlier analysis can be performed during the market intelligence phase.

### Data Integrity
- Do not automatically remove statistical outliers.
- Preserve original prices and volumes.
- Record separate price and volume outlier indicators.

In [9]:

# Create working copy of date-aligned stock dataset
df_stock_outliers = df_stocks_analysis.copy()

# Initialize outlier indicators
df_stock_outliers["price_outlier"] = False
df_stock_outliers["volume_outlier"] = False

outlier_summary = []

# Screen each stock independently
for symbol, group in df_stock_outliers.groupby("stock_symbol"):

    summary = {
        "stock_symbol": symbol,
        "total_records": len(group)
    }

    for column, flag_column, label in [
        ("close_price", "price_outlier", "price"),
        ("volume", "volume_outlier", "volume")
    ]:
        q1 = group[column].quantile(0.25)
        q3 = group[column].quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_mask = (
            (group[column] < lower_bound) |
            (group[column] > upper_bound)
        )

        df_stock_outliers.loc[
            group.index, flag_column
        ] = outlier_mask

        summary[f"{label}_outliers"] = int(
            outlier_mask.sum()
        )

        summary[f"{label}_outlier_pct"] = round(
            outlier_mask.mean() * 100, 2
        )

    outlier_summary.append(summary)

# Create summary report
df_stock_outlier_summary = pd.DataFrame(
    outlier_summary
)

# Flag records with either type of outlier
df_stock_outliers["is_outlier"] = (
    df_stock_outliers["price_outlier"] |
    df_stock_outliers["volume_outlier"]
)

# Extract flagged observations
df_stock_flagged_outliers = df_stock_outliers[
    df_stock_outliers["is_outlier"]
].copy()

print("Banking Stock Statistical Outlier Summary:")
display(df_stock_outlier_summary)

print("\nTotal Analysis Records:", len(df_stock_outliers))
print(
    "Closing Price Outliers:",
    int(df_stock_outliers["price_outlier"].sum())
)
print(
    "Trading Volume Outliers:",
    int(df_stock_outliers["volume_outlier"].sum())
)
print(
    "Unique Flagged Records:",
    len(df_stock_flagged_outliers)
)

if not df_stock_flagged_outliers.empty:
    print("\nFlagged Banking Stock Observations:")
    display(
        df_stock_flagged_outliers.sort_values(
            ["stock_symbol", "observation_date"]
        )
    )
else:
    print("\nNo statistical outliers detected.")

print("\nBanking Stock Statistical Outlier Screening: COMPLETED")


Banking Stock Statistical Outlier Summary:


,stock_symbol,total_records,price_outliers,price_outlier_pct,volume_outliers,volume_outlier_pct
0,HDFCBANK.BSE,99,0,0.0,2,2.02
1,ICICIBANK.BSE,99,0,0.0,10,10.10
2,SBIN.BSE,99,0,0.0,9,9.09



Total Analysis Records: 297
Closing Price Outliers: 0
Trading Volume Outliers: 21
Unique Flagged Records: 21

Flagged Banking Stock Observations:


,observation_date,stock_symbol,open_price,high_price,low_price,close_price,volume,source,price_outlier,volume_outlier,is_outlier
54,2026-08-10,HDFCBANK.BSE,731.55,737.40,728.30,732.00,6406231,Alpha Vantage,False,True,True
63,2026-08-21,HDFCBANK.BSE,728.10,732.85,726.75,727.50,8163779,Alpha Vantage,False,True,True
112,2026-06-11,ICICIBANK.BSE,1289.50,1333.00,1286.25,1317.40,4128427,Alpha Vantage,False,True,True
122,2026-06-25,ICICIBANK.BSE,1378.05,1404.55,1378.05,1387.90,1838904,Alpha Vantage,False,True,True
131,2026-07-09,ICICIBANK.BSE,1381.40,1403.25,1376.90,1381.60,2193131,Alpha Vantage,False,True,True
136,2026-07-16,ICICIBANK.BSE,1420.10,1424.00,1406.60,1418.20,2227737,Alpha Vantage,False,True,True
139,2026-07-21,ICICIBANK.BSE,1474.00,1474.00,1456.75,1462.95,1758118,Alpha Vantage,False,True,True
146,2026-07-30,ICICIBANK.BSE,1434.85,1445.00,1423.00,1436.50,2194319,Alpha Vantage,False,True,True
151,2026-08-06,ICICIBANK.BSE,1460.00,1476.95,1448.20,1476.95,2137432,Alpha Vantage,False,True,True
175,2026-09-09,ICICIBANK.BSE,1396.00,1401.70,1385.05,1390.10,2129133,Alpha Vantage,False,True,True



Banking Stock Statistical Outlier Screening: COMPLETED



## 9. Banking Stock Validation Summary and Processed Dataset Preparation

### Business Objective
Prepare a validated banking stock dataset for PostgreSQL integration, financial analysis, and Power BI reporting.

### Validation Findings
- Original stock observations: 300
- Date-aligned observations: 297
- Excluded out-of-period observations: 3
- Schema validation: PASSED
- Missing-value validation: PASSED
- Duplicate validation: PASSED
- OHLC and volume validation: PASSED
- Stock symbol and source validation: PASSED
- Date alignment after scope filtering: PASSED
- Closing-price outliers: 0
- Trading-volume outliers: 21

### Processing Decisions
1. Preserve the original 300-record raw dataset.
2. Retain only observations within the approved analysis period.
3. Preserve all 297 in-period observations.
4. Retain all 21 statistically flagged observations.
5. Add price_outlier, volume_outlier, and is_outlier indicators.
6. Save validation and audit reports separately.

### Business Interpretation
The banking stock dataset passed the required structural, completeness, uniqueness, and financial consistency checks.

The 21 flagged trading-volume observations require further investigation but are not confirmed data errors.

### Data Integrity
No original stock observations are deleted or overwritten.


In [10]:

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

# Define output directories
PROCESSED_DIR = PROJECT_ROOT / "Data" / "processed"
VALIDATION_DIR = PROJECT_ROOT / "Data" / "validation"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

# Prepare processed dataset
df_stocks_processed = (
    df_stock_outliers
    .sort_values(["stock_symbol", "observation_date"])
    .reset_index(drop=True)
    .copy()
)

# Final integrity checks
assert len(df_stocks_raw) == 300
assert len(df_stocks_processed) == 297
assert len(df_stock_outside_period) == 3

assert int(df_stocks_processed["price_outlier"].sum()) == 0
assert int(df_stocks_processed["volume_outlier"].sum()) == 21
assert int(df_stocks_processed["is_outlier"].sum()) == 21

assert not df_stocks_processed.duplicated(
    subset=STOCK_BUSINESS_KEY
).any()

assert df_stocks_processed["observation_date"].between(
    START_DATE, END_DATE
).all()

# Save processed dataset
processed_path = (
    PROCESSED_DIR / "banking_stocks_processed.csv"
)

df_stocks_processed.to_csv(
    processed_path,
    index=False
)

# Save statistical outlier review
outlier_path = (
    VALIDATION_DIR / "banking_stocks_outlier_review.csv"
)

df_stocks_processed[
    df_stocks_processed["is_outlier"]
].to_csv(
    outlier_path,
    index=False
)

# Save out-of-period observations
excluded_path = (
    VALIDATION_DIR / "banking_stocks_out_of_period.csv"
)

df_stock_outside_period.to_csv(
    excluded_path,
    index=False
)

# Build validation summary
validation_summary = {
    "dataset": "Indian Banking Stocks",
    "source": "Alpha Vantage",
    "validation_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "analysis_start_date": str(START_DATE.date()),
    "analysis_end_date": str(END_DATE.date()),
    "original_records": 300,
    "out_of_period_records": 3,
    "processed_records": 297,
    "records_by_stock": {
        symbol: int(count)
        for symbol, count in
        df_stocks_processed["stock_symbol"]
        .value_counts().items()
    },
    "schema_validation": "PASSED",
    "missing_value_validation": "PASSED",
    "duplicate_validation": "PASSED",
    "ohlc_volume_validation": "PASSED",
    "symbol_source_validation": "PASSED",
    "date_alignment_validation": "PASSED_AFTER_FILTERING",
    "price_outliers": 0,
    "volume_outliers": 21,
    "unique_flagged_records": 21,
    "outlier_review_status": "PENDING",
    "raw_records_deleted": 0
}

# Save validation summary
summary_path = (
    VALIDATION_DIR / "banking_stocks_validation_summary.json"
)

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(validation_summary, file, indent=4)

# Verify saved files
df_verified = pd.read_csv(processed_path)

assert len(df_verified) == 297
assert df_verified["is_outlier"].sum() == 21

print("Banking Stock Processed Dataset: SAVED")
print("Banking Stock Validation Summary: SAVED")
print("Banking Stock Outlier Review: SAVED")
print("Out-of-Period Records Report: SAVED")

print("\nOriginal Records:", len(df_stocks_raw))
print("Processed Records:", len(df_verified))
print("Out-of-Period Records:", len(df_stock_outside_period))
print("Flagged Outliers:", int(df_verified["is_outlier"].sum()))
print("Raw Records Deleted: 0")

print("\nProcessed Dataset Preview:")
display(df_verified.head())


Banking Stock Processed Dataset: SAVED
Banking Stock Validation Summary: SAVED
Banking Stock Outlier Review: SAVED
Out-of-Period Records Report: SAVED

Original Records: 300
Processed Records: 297
Out-of-Period Records: 3
Flagged Outliers: 21
Raw Records Deleted: 0

Processed Dataset Preview:


,observation_date,stock_symbol,open_price,high_price,low_price,close_price,volume,source,price_outlier,volume_outlier,is_outlier
0,2026-05-22,HDFCBANK.BSE,759.00,774.90,759.00,766.40,629420,Alpha Vantage,False,False,False
1,2026-05-25,HDFCBANK.BSE,778.00,788.00,775.25,786.85,964192,Alpha Vantage,False,False,False
2,2026-05-26,HDFCBANK.BSE,785.95,790.90,777.00,779.00,932064,Alpha Vantage,False,False,False
3,2026-05-27,HDFCBANK.BSE,772.10,772.10,756.60,758.50,4893862,Alpha Vantage,False,False,False
4,2026-05-29,HDFCBANK.BSE,749.00,760.65,738.10,744.75,1799471,Alpha Vantage,False,False,False


In [2]:

import pandas as pd

# Read the existing processed stock dataset.
stock_processed_path = (
    project_root / "Data" / "processed"
    / "banking_stocks_processed.csv"
)

stock_df = pd.read_csv(stock_processed_path)

# Calculate data quality metrics.
record_count = len(stock_df)
missing_values = int(stock_df.isna().sum().sum())

duplicate_count = int(
    stock_df.duplicated(
        subset=["observation_date", "stock_symbol"]
    ).sum()
)

price_outliers = int(stock_df["price_outlier"].sum())
volume_outliers = int(stock_df["volume_outlier"].sum())
total_flagged = int(stock_df["is_outlier"].sum())

# Record validation results.
logger.info("Banking stock validation results verified")
logger.info("Processed stock records: %s", record_count)
logger.info("Missing values: %s", missing_values)
logger.info("Duplicate stock-date records: %s", duplicate_count)
logger.info("Price outliers: %s", price_outliers)
logger.info("Volume outliers: %s", volume_outliers)
logger.info("Total flagged records: %s", total_flagged)

if (
    record_count == 297
    and missing_values == 0
    and duplicate_count == 0
):
    logger.info("Banking stock core data quality checks PASSED")
else:
    logger.error("Banking stock core data quality checks FAILED")


2026-11-02 16:07:40,982 | INFO | Banking_Stocks_Data_Validation | Banking stock validation results verified
2026-11-02 16:07:40,983 | INFO | Banking_Stocks_Data_Validation | Processed stock records: 297
2026-11-02 16:07:40,985 | INFO | Banking_Stocks_Data_Validation | Missing values: 0
2026-11-02 16:07:40,988 | INFO | Banking_Stocks_Data_Validation | Duplicate stock-date records: 0
2026-11-02 16:07:40,989 | INFO | Banking_Stocks_Data_Validation | Price outliers: 0
2026-11-02 16:07:40,990 | INFO | Banking_Stocks_Data_Validation | Volume outliers: 21
2026-11-02 16:07:40,991 | INFO | Banking_Stocks_Data_Validation | Total flagged records: 21
2026-11-02 16:07:40,992 | INFO | Banking_Stocks_Data_Validation | Banking stock core data quality checks PASSED
